<a href="https://colab.research.google.com/github/photominion777/exposure-value-to-light-value/blob/main/raw-meter/raw_meter_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# @title
# ===================================================================================================
# LINEAR RAW EXPOSURE METER SIMULATOR (PART 1: LOGIC & COMPUTATION ENGINE)
# ===================================================================================================
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Global option lists representing the manufacturer display values
N_OPTIONS = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.5, 2.8, 3.2, 3.5, 4.0, 4.5, 5.0, 5.6, 6.3, 7.1, 8.0, 9.0, 10, 11, 13, 14, 16, 18, 20, 22, 25, 29, 32]
T_OPTIONS = ["1", "1/1.3", "1/1.6", "1/2", "1/2.5", "1/3.2", "1/4", "1/5", "1/6", "1/8", "1/10", "1/13", "1/15", "1/20", "1/25", "1/30", "1/40", "1/50", "1/60", "1/80", "1/100", "1/125", "1/160", "1/200", "1/250", "1/320", "1/400", "1/500", "1/640", "1/800", "1/1000", "1/1250", "1/1600", "1/2000", "1/2500", "1/3200", "1/4000", "1/5000", "1/6400", "1/8000"]
S_OPTIONS = [50, 64, 80, 100, 125, 160, 200, 250, 320, 400, 500, 640, 800, 1000, 1250, 1600, 2000, 2500, 3200, 4000, 5000, 6400, 8000, 10000, 12800, 16000, 20000, 25600, 51200, 102400]
BIT_DEPTH_OPTIONS = [12, 14, 16]

# Container and dynamic environment label
output_area = widgets.Output()
lighting_text_right = widgets.Label(value="Sunny", layout=widgets.Layout(width='250px', margin='0 0 0 10px'))

def get_lighting_name(lv):
    ranges = [
        (19.5, 22.5, "Industrial Laser / Lab Light"), (18.5, 19.5, "Arc Welding / High-Power LED"),
        (17.0, 18.5, "Studio Flash / Searchlight"), (15.5, 17.0, "Extreme Sun (Snow/Sand)"),
        (14.5, 15.5, "Sunny"), (13.5, 14.5, "Hazy Sun"), (12.5, 13.5, "Bright Overcast"),
        (11.5, 12.5, "Overcast / Cloudy"), (10.5, 11.5, "Deep Shade"), (9.5, 10.5, "Sunset / Sunrise"),
        (8.5, 9.5, "Very Dynamic Twilight"), (7.5, 8.5, "Bright Street Lighting"),
        (6.5, 7.5, "Blue Hour / City Night"), (5.5, 6.5, "Bright Indoor"), (4.5, 5.5, "Standard Indoor"),
        (3.5, 4.5, "Living Room (Evening)"), (2.5, 3.5, "Dim Indoor"), (1.5, 2.5, "Distant Building Lights"),
        (0.5, 1.5, "Very Dim Interior"), (-0.5, 0.5, "Night Sky / City Skyline"),
        (-1.5, -0.5, "Dim Night Street"), (-2.5, -1.5, "Full Moon (Snow)"),
        (-3.5, -2.5, "Full Moon (Landscape)"), (-4.5, -3.5, "Quarter Moon"),
        (-5.5, -4.5, "Crescent Moon"), (-7.5, -5.5, "Starlight Night")
    ]
    for low, high, name in ranges:
        if low <= lv < high: return name
    return "Transition / Mixed Light"

def calculate_and_display_raw(L_input, N, t_str, S, bit_depth, ec_index):
    idx_N, idx_t, idx_S = N_OPTIONS.index(N), T_OPTIONS.index(t_str), S_OPTIONS.index(S)
    N_exact = 1.0 * (2**(1/6))**(idx_N - 3)
    t_exact = 1.0 * (2**(-1/3))**idx_t
    S_exact = 100.0 * (2**(1/3))**(idx_S - 3)

    L_eff_peak = float(L_input)
    K = 12.5

    # Definition from 1st article: LV_ext = log2(100 * L_eff / K)
    LV_ext = math.log2((100.0 * L_eff_peak) / K) if L_eff_peak > 0 else -5.0
    lighting_text_right.value = get_lighting_name(LV_ext)

    # 2. Setup fixed physical sensor boundaries and APEX log variables
    Y_sat = (2**bit_depth) - 1            # Constant Hardware Wall
    y_sat = math.log2(Y_sat)              # Constant Log Ceiling

    av = math.log2(N_exact**2)            # Aperture Value
    tv = math.log2(t_exact)               # Time Value
    sv_raw = math.log2(S_exact / 100.0)   # Sensor Gain (Equation 1, G(S))

    analog_hardware_base = 3.00           # Baseline constant for the pure substrate mapping
    kappa_log2 = analog_hardware_base + math.log2(2**bit_depth - 1)
    photometric_correction = math.log2(K / 100.0)

    # formula: e_peak = lb(kappa) + LV_ext + lb(K/100)
    e_peak = kappa_log2 + LV_ext + photometric_correction
    y_peak = e_peak - av + tv + sv_raw
    Y_peak_linear = 2**y_peak

    # 3. The deviation as (-1) times headroom and the shift by user Artistic Bias
    delta_y_ETTR = y_peak - y_sat
    ec_user = ec_index * (1.0 / 3.0)
    delta_y_Display = delta_y_ETTR - ec_user

    # 4. Build the ASCII viewfinder scale matrix
    scale_header = " -5       -4       -3       -2       -1        0       +1       +2       +3       +4       +5  "
    scale_ticks = []
    for index in range(-15, 16):
        tick_value = index / 3.0
        if abs(delta_y_Display - tick_value) < (1.0 / 6.0) and -5.1 < delta_y_Display < 5.1:
            scale_ticks.append("▲")
        elif index % 3 == 0: scale_ticks.append("|")
        else: scale_ticks.append("·")

    scale_visual = ("◀ " if delta_y_Display < -5.1 else "  ") + "  ".join(scale_ticks) + (" ▶" if delta_y_Display > 5.1 else "  ")
    clipping_ratio = (Y_peak_linear / Y_sat) * 100.0

    if Y_peak_linear <= Y_sat:
        percentage_pixel_clipping = 0.0
    else:
        x_vals_calc = np.linspace(0, Y_peak_linear * 1.5, 5000)
        sigma_calc = Y_peak_linear * 0.22
        y_hist_calc = np.exp(-((x_vals_calc - Y_peak_linear * 0.8)**2) / (2 * sigma_calc**2)) * 0.7 + \
                      np.exp(-((x_vals_calc - Y_peak_linear * 0.3)**2) / (2 * (sigma_calc * 1.5)**2)) * 0.4 + \
                      np.exp(-((x_vals_calc - Y_peak_linear)**2) / (2 * (sigma_calc * 0.05)**2)) * 0.3

        total_area = np.trapezoid(y_hist_calc, x_vals_calc)
        clipped_mask = x_vals_calc > Y_sat

        if np.any(clipped_mask) and total_area > 0:
            clipped_area = np.trapezoid(y_hist_calc[clipped_mask], x_vals_calc[clipped_mask])
            percentage_pixel_clipping = (clipped_area / total_area) * 100.0
        else:
            percentage_pixel_clipping = 0.0

    total_sensor_pixels = 24000000
    absolute_clipped_pixels = int((percentage_pixel_clipping / 100.0) * total_sensor_pixels)

    render_dashboard_view(bit_depth, Y_sat, Y_peak_linear, LV_ext, clipping_ratio, delta_y_ETTR, ec_user, delta_y_Display, scale_header, scale_visual, absolute_clipped_pixels, total_sensor_pixels, percentage_pixel_clipping)

# @title LINEAR RAW EXPOSURE METER SIMULATOR
# ===================================================================================================
# LINEAR RAW EXPOSURE METER SIMULATOR (PART 2: DASHBOARD RENDERING & UI COUPLING)
# ===================================================================================================
def render_dashboard_view(bit_depth, Y_sat, Y_peak_linear, LV_ext, clipping_ratio, delta_y_ETTR, ec_user, delta_y_Display, scale_header, scale_visual, absolute_clipped_pixels, total_sensor_pixels, percentage_pixel_clipping):
    # Format absolute pixel values into a clean, compact Mpx format for both terminal and plot
    clipped_mpx = absolute_clipped_pixels / 1_000_000
    total_mpx = total_sensor_pixels / 1_000_000

    output_text = f"""===================================================================================================
[ INTERNAL LINEAR RAW METER SIMULATION - RSI PARSING ENGINE ]
===================================================================================================
Sensor ADC Bit-Depth Resolution    :  {bit_depth}-bit Domain
Absolute Saturation Limit (Y_sat)  :  {Y_sat:,} Digital Numbers (DN) [Constant Hardware Ceiling]
Peak Linear Channel Signal (Y_peak):  {min(int(Y_peak_linear), Y_sat):,} DN {"[🚨 CLIPPED AT HARDWARE WALL]" if Y_peak_linear > Y_sat else ""}
---------------------------------------------------------------------------------------------------
RAW Sensor Log Ceiling (y_sat)     :  {math.log2(Y_sat):.2f} stops [Constant Hardware Baseline]
Environmental Light Value (LV_ext) :  {LV_ext:.2f}
ADC Range Utilization              :  {clipping_ratio:.1f}%
---------------------------------------------------------------------------------------------------
[ERGONOMIC LIVE PIXEL CLIPPING METRICS]
Absolute Pixel Clipping            :  {clipped_mpx:.2f} Mpx / {total_mpx:.0f} Mpx
Percentage Pixel Clipping          :  {percentage_pixel_clipping:.2f} % of Sensor Area
---------------------------------------------------------------------------------------------------
True RAW Signal Index (RSI_peak)   :  {delta_y_Display:+.2f} Stops (Normed: 0 RSI = 12.5% Saturation)
User Artistic Override (EC_user)   :  {ec_user:+.2f} Stops
---------------------------------------------------------------------------------------------------
Displayed RAW Viewfinder Index (RSI_Display): {delta_y_Display:+.2f} Stops

{scale_header}
{scale_visual}
===================================================================================================
"""
    if Y_peak_linear > Y_sat:
        output_text += f"🚨 CRITICAL HARDWARE CLIPPING: Exceeds sensor capacity by {delta_y_ETTR:.2f} f-stops.\n"
    elif abs(delta_y_ETTR) < 0.01:
        output_text += "✅ OPTIMAL ETTR EXPOSURE: Perfect sensor capacity utilization."
    else:
        output_text += f"⚠️ UNDERUTILIZED DYNAMIC RANGE: Leaving {abs(delta_y_ETTR):.2f} f-stops of hardware headroom un-filled."

    with output_area:
        clear_output(wait=True)
        display(HTML(f"<pre style='font-family: Courier New; line-height: 1.2; font-size: 14px; background-color: #1e1e1e; color: #f0f0f0; padding: 15px; border-radius: 5px;'>{output_text}</pre>"))

        fig, ax1 = plt.subplots(figsize=(11, 4.5))
        x_vals = np.logspace(math.log2(256), math.log2(262140), 1500, base=2)

        # Shape moves naturally according to TRUE Y_peak_linear
        sigma = Y_peak_linear * 0.22
        y_hist = np.exp(-((x_vals - Y_peak_linear * 0.8)**2) / (2 * sigma**2)) * 0.7 + \
                 np.exp(-((x_vals - Y_peak_linear * 0.3)**2) / (2 * (sigma * 1.5)**2)) * 0.4 + \
                 np.exp(-((x_vals - Y_peak_linear)**2) / (2 * (sigma * 0.05)**2)) * 0.3

        # In clipping region, i.e. on the right side of Y_sat data points will be set to zero
        if Y_peak_linear > Y_sat:
            y_hist[x_vals > Y_sat] = 0.0

        # Plot valid shaded regions (Green)
        ax1.fill_between(x_vals, 0, y_hist, where=(x_vals <= Y_sat), color='#2ca02c', alpha=0.6, label=f'Valid RAW Sensor Data ({100.0 - percentage_pixel_clipping:.2f}%)')

        # SATURATION BIN FIX
        # Clipped pixels are drawn as red bar, its height correlates with the % of clipped pixels
        if percentage_pixel_clipping > 0.01:
            clip_box_x = x_vals[(x_vals >= Y_sat * 0.95) & (x_vals <= Y_sat)]
            clip_box_height = 0.1 + (percentage_pixel_clipping / 100.0) * 0.9
            ax1.fill_between(clip_box_x, 0, clip_box_height, color='#d62728', alpha=0.85, label='CRITICAL: Clipping (Data Loss)')

        # Dynamic Clipping Indicators (triangles) instead of solid line
        ax1.scatter(Y_sat, 0.03, color='black', marker='^', s=120, zorder=5, label='Hardware Ceiling (Y_sat)')
        ax1.scatter(Y_sat, 1.07, color='black', marker='v', s=120, zorder=5, clip_on=False)

        ax1.set_xscale('log', base=2)
        dn_ticks = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144]
        dn_labels = ['256', '512', '1.024', '2.048', '4.096', '8.192', '16.384', '32.767', '65.535', '131.070', '262.140']
        ax1.set_xticks(dn_ticks)
        ax1.set_xticklabels(dn_labels, fontsize=9)
        ax1.set_xlim(256, 262140)
        ax1.set_ylim(0, 1.1)
        ax1.set_xlabel('Linear Digital Signal Level [Digital Numbers (DN)]', fontsize=10, labelpad=8)
        ax1.set_ylabel('Pixel Distribution Frequency', fontsize=10, labelpad=8)
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.legend(loc='upper left', frameon=True)

        ax2 = ax1.twiny()
        ax2.set_xscale('log', base=2)
        ax2.set_xlim(ax1.get_xlim())
        ax2.set_xticks(dn_ticks)
        rsi_labels = ['-5 RSI', '-4 RSI', '-3 RSI', '-2 RSI', '-1 RSI', '0 RSI', '+1 RSI', '+2 RSI', '+3 RSI', '+4 RSI', '+5 RSI']
        ax2.set_xticklabels(rsi_labels, fontsize=9, color='#2980b9')
        ax2.set_xlabel('Logarithmic Raw Signal Index [RSI] Scale in Stops (Top)', fontsize=10, color='#2980b9', labelpad=8)

        plt.title(f'VISUAL SENSOR DASHBOARD ({bit_depth}-bit Domain)\nLogarithmic Raw Level [RSI] vs. Physical Signal Amplitude (DN)', fontsize=12, fontweight='bold', pad=15)

        # FIX: percentage_pixel_clipping formatted to .2f for clean presentation
        info_text = f"Clipped Pixels:\n{clipped_mpx:.2f} Mpx / {total_mpx:.0f} Mpx\n({percentage_pixel_clipping:.2f}%)"
        props = dict(boxstyle='round,pad=0.3', facecolor='#d62728' if percentage_pixel_clipping > 0 else '#2ca02c', alpha=0.15, edgecolor='none')
        ax1.text(Y_sat * 1.06, 0.82, info_text, fontsize=9, fontweight='bold', color='#c0392b' if percentage_pixel_clipping > 0 else '#27ae60', ha='left', va='center', bbox=props)

        plt.tight_layout()
        plt.show()

# Setup UI Layout and Interactive Sliders
radiance_options = list()
for index in range(-18, 67):
    l_phys = (2**(index/3.0)) / (100 / 12.5)
    l_phys = round(l_phys, 1) if l_phys >= 10 else round(l_phys, 2)
    if l_phys not in radiance_options: radiance_options.append(l_phys)

slider_layout, style = widgets.Layout(width='450px'), {'description_width': '140px'}
l_slider = widgets.SelectionSlider(options=radiance_options, value=4096.0, description="Scene Radiance:", readout=False, layout=slider_layout, style=style)
n_slider = widgets.SelectionSlider(options=N_OPTIONS, value=16, description="Aperture (N):", layout=slider_layout, style=style)
t_slider = widgets.SelectionSlider(options=T_OPTIONS, value="1/125", description="Shutter (t):", layout=slider_layout, style=style)
s_slider = widgets.SelectionSlider(options=S_OPTIONS, value=100, description="ISO Speed (S):", layout=slider_layout, style=style)
bit_dropdown = widgets.Dropdown(options=BIT_DEPTH_OPTIONS, value=16, description="Sensor Bit-Depth:", layout=widgets.Layout(width='250px'), style=style)
ec_slider = widgets.IntSlider(min=-9, max=9, step=1, value=0, description="Artistic Bias (EC):", readout=False, layout=slider_layout, style=style)

ec_readout = widgets.Label(value="0.00 stops", layout=widgets.Layout(margin='0 0 0 10px'))
def update_ec_label(change): ec_readout.value = f"{change['new']*(1.0/3.0):+.2f} stops"
ec_slider.observe(update_ec_label, names='value')

ui_controls = widgets.interactive_output(calculate_and_display_raw, {
    'L_input': l_slider, 'N': n_slider, 't_str': t_slider, 'S': s_slider, 'bit_depth': bit_dropdown, 'ec_index': ec_slider
})
display(widgets.VBox([widgets.HBox([l_slider, lighting_text_right]), n_slider, t_slider, s_slider, widgets.HBox([ec_slider, ec_readout]), widgets.HBox([bit_dropdown])]), output_area)


Output()